In [1]:
# Pasa 1: instalar librerías
!pip install openai PyMuPDF --quiet


In [15]:
%pip install --upgrade PyMuPDF openai


Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install PyMuPDF openai


Note: you may need to restart the kernel to use updated packages.


In [17]:
import fitz
print("✅ PyMuPDF (fitz) importado correctamente, versión:", fitz.__doc__.split()[1])
from openai import OpenAI
print("✅ openai importado correctamente.")


✅ PyMuPDF (fitz) importado correctamente, versión: 1.26.0:
✅ openai importado correctamente.


In [21]:
# —————————————————————————————————————
# Celda 1: Instalar PyMuPDF y openai en el kernel activo
# —————————————————————————————————————
%pip install --upgrade pip PyMuPDF openai --quiet


Note: you may need to restart the kernel to use updated packages.


In [23]:
# —————————————————————————————————————
# Celda 2: Importar librerías necesarias
# —————————————————————————————————————
import fitz            # PyMuPDF
from openai import OpenAI
from getpass import getpass
import json
import pandas as pd

# Verificar que fitz se importó correctamente:
print("✅ PyMuPDF (fitz) importado. Versión:", fitz.__doc__.split()[1])
print("✅ openai importado correctamente.")


✅ PyMuPDF (fitz) importado. Versión: 1.26.0:
✅ openai importado correctamente.


In [33]:
import os
print(os.getcwd())          # Te dice la carpeta actual de trabajo
print(os.listdir())         # Te listará todos los archivos en esa carpeta


C:\Users\ARIANA
['.anaconda', '.cache', '.conda', '.condarc', '.continuum', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.matplotlib', '.ms-ad', '.wdm', '.xlwings', '237622_hw2_2025_1.ipynb', '3.1 ARIANA GUTIERREZ.ipynb', '3D Objects', 'ado', 'anaconda3', 'anaconda_projects', 'AppData', 'Application Data', 'bvl_2025-04-10.xlsx', 'bvl_diario.xlsx', 'CODIGO.ipynb', 'Configuración local', 'Contacts', 'Contests', 'Cookies', 'Datos de programa', 'Documents', 'Downloads', 'Entorno de red', 'Favorites', 'filtered_codeforces_contests.csv', 'IIGeo_24-47_002_Arevalo.pdf', 'Impresoras', 'Links', 'Menú Inicio', 'Microsoft', 'MicrosoftEdgeBackups', 'Mis documentos', 'Music', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{53b39e88-18c4-11ea-a811-000d3aa4692b}.TM.blf', 'NTUSER.DAT{53b39e88-18c4-11ea-a811-000d3aa4692b}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{53b39e88-18c4-11ea-a811-000d3aa4692b}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.in

In [35]:
pdf_path = "IIGeo_24-47_002_Arevalo.pdf"


In [37]:
# —————————————————————————————————————
# Celda 3 (modificada): Definir ruta del PDF y extraer texto
# —————————————————————————————————————

pdf_path = "IIGeo_24-47_002_Arevalo.pdf"  # Ahora apuntamos al PDF que subiste

def extraer_texto_pdf(ruta_pdf: str) -> str:
    """
    Abre el PDF con PyMuPDF (fitz) y concatena todo el texto de cada página.
    """
    doc = fitz.open(ruta_pdf)
    texto = ""
    for pagina in doc:
        texto += pagina.get_text()
    doc.close()
    return texto

# Ejecuta la extracción:
texto_paper = extraer_texto_pdf(pdf_path)
print("✅ Extracción finalizada. Primeros 300 caracteres:\n")
print(texto_paper[:300])

# Guardar el texto extraído en un archivo .txt
with open("paper_tree_cover_north_lima.txt", "w", encoding="utf-8") as f:
    f.write(texto_paper)
print("✅ Texto guardado en 'paper_tree_cover_north_lima.txt'")


✅ Extracción finalizada. Primeros 300 caracteres:

© Los autores. Este artículo es publicado por la Revista del Instituto de investigación de la Facultad de minas, metalurgia y ciencias 
geográficas de la Universidad Nacional Mayor de San Marcos. Este es un artículo de acceso abierto, distribuido bajo los términos de la 
licencia Creative Commons At
✅ Texto guardado en 'paper_tree_cover_north_lima.txt'


In [69]:
# —————————————————————————————————————
# Celda 4: Pedir API Key y configurar OpenAI (versión ≥ 1.0.0)
# —————————————————————————————————————

import openai
from getpass import getpass

# 1) Pedir la API key (no se mostrará en pantalla)
api_key = getpass("🔐 Ingresa tu API key de OpenAI: ").strip()

# 2) Verificar que empiece con "sk-"
if not api_key.startswith("sk-"):
    raise ValueError("‼️ La API key no parece válida: debe empezar con 'sk-'.")
openai.api_key = api_key

# 3) Hacer una llamada mínima de prueba para chequear que la clave funciona
try:
    prueba = openai.chat.completions.create(
        model="gpt-3.5-turbo",            # modelo de prueba
        messages=[
            {"role": "system",  "content": "You are a friendly assistant."},
            {"role": "user",    "content": "Ping"}
        ],
        temperature=0
    )
    print("✅ Cliente de OpenAI configurado correctamente.")
    # (Opcional) Si quieres ver la respuesta de “Ping”:
    # print("↪ Respuesta de prueba:", prueba.choices[0].message.content)
except Exception as e:
    raise RuntimeError(
        "❌ Falló la autenticación. Verifica que tu clave esté activa y sin espacios extras.\n"
        f"Detalle técnico: {e}"
    )




🔐 Ingresa tu API key de OpenAI:  ········


✅ Cliente de OpenAI configurado correctamente.


In [71]:
# —————————————————————————————————————
# Celda 5: Ranking de eficiencia de captura de CO₂
# —————————————————————————————————————
import json
import pandas as pd

prompt_ranking = f"""
Eres un analista experto en captura de carbono urbana.
…(resto del prompt)…
Texto completo del paper:
\"\"\"{texto_paper}\"\"\"
"""

respuesta_ranking = openai.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "Eres un analista experto en captura de carbono urbana."},
        {"role": "user",   "content": prompt_ranking}
    ],
    temperature=0
)

ranking_json = respuesta_ranking.choices[0].message.content.strip()
print("📊 Ranking (JSON):\n", ranking_json)

try:
    ranking_list = json.loads(ranking_json)
    print("✅ JSON parseado correctamente.")
    df_ranking = pd.DataFrame(ranking_list)
    display(df_ranking)
    df_ranking.to_csv("ranking_eficiencia_parques.csv", index=False, encoding="utf-8")
    print("✅ CSV guardado.")
except Exception as e:
    print("⚠️ No se pudo parsear el JSON. Detalle:", e)


📊 Ranking (JSON):
 Este artículo de investigación se centra en la relación entre la cobertura arbórea y la captura de dióxido de carbono en los parques urbanos de Lima Norte. Los autores utilizaron un enfoque cuantitativo y correlacional para su estudio, utilizando trabajo de campo y remoto para recopilar datos.

Los resultados mostraron una correlación de Pearson de 0.994, lo que indica una relación positiva muy fuerte entre la cobertura arbórea y la captura de carbono en los parques del área de estudio. Sin embargo, los valores obtenidos en relación con el volumen de captura de dióxido de carbono (0.34 CO2/m2-año) fueron muy bajos en comparación con los resultados de investigaciones realizadas en otras ciudades.

Los autores concluyen que la vegetación arbórea es importante e influye de manera significativa en la función de regulación de los parques urbanos. Recomiendan la selección de especies de árboles más adecuadas para mejorar la funcionalidad de los parques en la sostenibilidad

In [73]:
# —————————————————————————————————————
# Celda 6: Prompt para recomendar especies arbóreas óptimas
# —————————————————————————————————————
import json

prompt_especies = f"""
Eres un dendrólogo urbano y consultor ambiental.

Con base en el texto del paper “Tree Cover and Carbon Dioxide Capture in Urban Parks: The Case of Northern Lima” y considerando las condiciones climáticas y espaciales de Lima Norte, 
recomienda **hasta tres especies de árboles** óptimas para cada uno de los parques evaluados, con el objetivo de **maximizar la captura anual de CO₂**.

Entrega únicamente un **JSON** con esta estructura EXACTA (sin texto adicional ni explicaciones):
{{
  "<nombre del parque>": [
    {{
      "Especie": "<nombre científico>",
      "Razonamiento": "<breve explicación de por qué es óptima>"
    }},
    ... (hasta 3 especies)
  ],
  ...
}}

Texto completo del paper:
\"\"\"{texto_paper}\"\"\"
"""

respuesta_especies = openai.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "Eres un dendrólogo urbano y consultor ambiental."},
        {"role": "user",   "content": prompt_especies}
    ],
    temperature=0.2
)

# 1) Extraer el texto (JSON-formato) de la respuesta
especies_json = respuesta_especies.choices[0].message.content.strip()
print("🌱 Recomendaciones de especies (JSON):\n")
print(especies_json)

# 2) Intentar parsear el JSON de especies
try:
    especies_dict = json.loads(especies_json)
    print("\n✅ El JSON de especies se parseó correctamente.")

    # 3) Guardar en un archivo .json
    with open("recomendaciones_especies.json", "w", encoding="utf-8") as f:
        json.dump(especies_dict, f, ensure_ascii=False, indent=2)
    print("✅ Recomendaciones guardadas en 'recomendaciones_especies.json'.")
except Exception as e:
    especies_dict = None
    print("⚠️ No se pudo parsear el JSON de especies:", e)


🌱 Recomendaciones de especies (JSON):

{
  "Parque CO1": [
    {
      "Especie": "Schinus molle",
      "Razonamiento": "Esta especie es resistente a la sequía y tiene una alta tasa de captura de CO2."
    },
    {
      "Especie": "Schinus terebinthifolius",
      "Razonamiento": "Esta especie es resistente a la sequía y tiene una alta tasa de captura de CO2."
    },
    {
      "Especie": "Grevillea robusta",
      "Razonamiento": "Esta especie es resistente a la sequía y tiene una alta tasa de captura de CO2."
    }
  ],
  "Parque CO2": [
    {
      "Especie": "Schinus molle",
      "Razonamiento": "Esta especie es resistente a la sequía y tiene una alta tasa de captura de CO2."
    },
    {
      "Especie": "Eucalyptus globulus",
      "Razonamiento": "Esta especie es resistente a la sequía y tiene una alta tasa de captura de CO2."
    },
    {
      "Especie": "Ficus benjamina",
      "Razonamiento": "Esta especie es resistente a la sequía y tiene una alta tasa de captura de CO2

In [75]:
# —————————————————————————————————————
# Celda 7: Prompt para diagnóstico distrital
# —————————————————————————————————————
import json
import pandas as pd

prompt_distritos = f"""
Eres un analista de políticas públicas ambientales.

A partir del texto del paper “Tree Cover and Carbon Dioxide Capture in Urban Parks: The Case of Northern Lima”, genera un **diagnóstico por distrito** (por ejemplo, Independencia, Comas, Los Olivos, etc.). Para cada distrito, indica:
1. Area_parques_ha (valor numérico).
2. Cobertura_arboŕea_promedio_%_o_CO2_total (valor numérico).
3. Retos (breve descripción de los principales problemas en el distrito).
4. Oportunidades (breve descripción de posibles acciones o ventajas).

Devuelve únicamente un **JSON** con esta estructura EXACTA:
[
  {{
    "Distrito": "<nombre del distrito>",
    "Area_parques_ha": "<valor numérico>",
    "Cobertura_arboŕea_%_o_CO2_total": "<valor numérico>",
    "Retos": "<breve descripción>",
    "Oportunidades": "<breve descripción>"
  }},
  ...
]

Texto completo del paper:
\"\"\"{texto_paper}\"\"\"
"""

respuesta_distritos = openai.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "Eres un analista de políticas públicas ambientales."},
        {"role": "user",   "content": prompt_distritos}
    ],
    temperature=0
)

# 1) Extraer el JSON de la respuesta
distritos_json = respuesta_distritos.choices[0].message.content.strip()
print("📍 Diagnóstico distrital (JSON):\n")
print(distritos_json)

# 2) Intentar parsear ese JSON
try:
    distritos_list = json.loads(distritos_json)
    print("\n✅ El JSON de distritos se parseó correctamente.")

    # 3) Convertir a DataFrame y mostrar
    df_distritos = pd.DataFrame(distritos_list)
    print("\n📋 DataFrame de diagnóstico distrital:")
    display(df_distritos)

    # 4) Guardar en un CSV
    df_distritos.to_csv("diagnostico_distrital.csv", index=False, encoding="utf-8")
    print("✅ Diagnóstico distrital guardado en 'diagnostico_distrital.csv'.")
except Exception as e:
    distritos_list = None
    print("⚠️ No se pudo parsear el JSON de distritos:", e)


📍 Diagnóstico distrital (JSON):

[
  {
    "Distrito": "Independencia",
    "Area_parques_ha": 2500,
    "Cobertura_arboŕea_%_o_CO2_total": 70,
    "Retos": "A pesar de tener un alto porcentaje de cobertura arbórea, el distrito de Independencia tiene un área de parques relativamente baja. Esto puede limitar la capacidad del distrito para capturar CO2 y mejorar la calidad del aire.",
    "Oportunidades": "El distrito tiene la oportunidad de aumentar la cantidad de parques y áreas verdes para mejorar la captura de CO2. Además, puede implementar políticas para proteger y mantener su alta cobertura arbórea."
  },
  {
    "Distrito": "Comas",
    "Area_parques_ha": 3100,
    "Cobertura_arboŕea_%_o_CO2_total": 12,
    "Retos": "El distrito de Comas tiene un bajo porcentaje de cobertura arbórea y una cantidad moderada de parques. Esto puede limitar su capacidad para capturar CO2 y mejorar la calidad del aire.",
    "Oportunidades": "El distrito tiene la oportunidad de aumentar su cobertura 

,Distrito,Area_parques_ha,Cobertura_arboŕea_%_o_CO2_total,Retos,Oportunidades
0,Independencia,2500,70,A pesar de tener un alto porcentaje de cobertu...,El distrito tiene la oportunidad de aumentar l...
1,Comas,3100,12,El distrito de Comas tiene un bajo porcentaje ...,El distrito tiene la oportunidad de aumentar s...
2,Los Olivos,2800,50,Aunque Los Olivos tiene un porcentaje moderado...,Los Olivos tiene la oportunidad de implementar...
3,San Martín de Porres,7000,51,San Martín de Porres tiene un porcentaje moder...,El distrito tiene la oportunidad de implementa...


✅ Diagnóstico distrital guardado en 'diagnostico_distrital.csv'.


In [77]:
import pandas as pd

# 1) Ruta al archivo CSV (si está en el mismo directorio donde corres el notebook, basta con el nombre)
ruta_csv = "diagnostico_distrital.csv"

# 2) Leer el CSV en un DataFrame
df_distritos = pd.read_csv(ruta_csv)

# 3) Mostrar las primeras filas para verificar que se cargó correctamente
print("Primeras filas de diagnóstico_distrital.csv:\n")
print(df_distritos.head())


Primeras filas de diagnóstico_distrital.csv:

               Distrito  Area_parques_ha  Cobertura_arboŕea_%_o_CO2_total  \
0         Independencia             2500                                70   
1                 Comas             3100                                12   
2            Los Olivos             2800                                50   
3  San Martín de Porres             7000                                51   

                                               Retos  \
0  A pesar de tener un alto porcentaje de cobertu...   
1  El distrito de Comas tiene un bajo porcentaje ...   
2  Aunque Los Olivos tiene un porcentaje moderado...   
3  San Martín de Porres tiene un porcentaje moder...   

                                       Oportunidades  
0  El distrito tiene la oportunidad de aumentar l...  
1  El distrito tiene la oportunidad de aumentar s...  
2  Los Olivos tiene la oportunidad de implementar...  
3  El distrito tiene la oportunidad de implementa...  
